In [2]:
import pandas as pd, numpy as np, time
from nltk.sentiment import SentimentIntensityAnalyzer
from textblob import TextBlob
import nltk, os
nltk.download('vader_lexicon', quiet=True)
os.chdir(os.path.expanduser('~/capstone/discovery-lens'))

df = pd.read_csv('notebooks/chunks_for_labelling_sample.csv')
df['label'] = df['label'].astype(str).replace('nan', '')
texts = df['text'].tolist()
true_labels = df['label'].tolist()

sia = SentimentIntensityAnalyzer()
vader_scores = [sia.polarity_scores(t)['compound'] for t in texts]
vader_labels = ['positive' if s >= 0.05 else 'negative' if s <= -0.05 else 'neutral' for s in vader_scores]
vader_acc = sum(p == t for p, t in zip(vader_labels, true_labels)) / len(true_labels)

tb_scores = [TextBlob(t).sentiment.polarity for t in texts]
tb_labels = ['positive' if s >= 0.05 else 'negative' if s <= -0.05 else 'neutral' for s in tb_scores]
tb_acc = sum(p == t for p, t in zip(tb_labels, true_labels)) / len(true_labels)

print(f'VADER    accuracy={vader_acc:.2%} spread={np.std(vader_scores):.4f}')
print(f'TextBlob accuracy={tb_acc:.2%} spread={np.std(tb_scores):.4f}')

VADER    accuracy=38.33% spread=0.4398
TextBlob accuracy=41.67% spread=0.2509


## Sentiment Model Benchmark
**Date:** May 13 2026  
**Author:** Lucas  
**Branch:** feature/sentiment-benchmark

### Goal
VADER reads calm professional text as neutral, collapsing odi_score discriminating
power on B2B corpora. This notebook benchmarks free CPU-runnable alternatives to
find a replacement.

### Ground truth
60 manually labelled chunks sampled from synthetic data:
- 20 interview chunks (Revolut, Asana, Lidl Plus)
- 20 support ticket chunks
- 20 review chunks

Label distribution: 29 negative / 16 neutral / 15 positive

Labelling criteria:
- negative: frustration, complaints, workarounds, unmet expectations — including
  calm or professional expressions of dissatisfaction
- positive: satisfaction, praise, things working well
- neutral: purely descriptive, no emotional signal

File: notebooks/chunks_for_labelling_sample.csv

## Benchmark Results

All models run on the same 60 ground truth chunks.

| Model | Accuracy | Score Spread (std dev) | Inference time | Neutral class |
|-------|----------|------------------------|----------------|---------------|
| VADER (baseline) | 38.33% | 0.4398 | ~0.06s | ✅ |
| TextBlob | 41.67% | 0.2509 | ~0.24s | ✅ |
| DistilBERT SST-2 | 53.33% | 0.9745 | ~2.1s | ❌ |
| lxyuan multilingual | 58.33% | 0.6446 | ~2.1s | ✅ |

### Why spread matters for this pipeline
`satisfaction` is derived directly from sentiment score via `(score + 1) / 2`.
If all scores cluster around 0, all clusters get satisfaction ≈ 0.5, which means
`odi_score = importance × 0.5` for everything — the unmet need signal disappears
and odi_score becomes a proxy for cluster size only.

High spread means the model is confidently differentiating between satisfied and
frustrated chunks, which produces meaningful odi_score variation across clusters.

### Notes on SST-2
- Best spread (0.9745) but no neutral class — every neutral chunk is forced into
  positive or negative, structurally penalising accuracy on this 3-class corpus
- PyTorch bus error on arm64 (Apple Silicon) — required TensorFlow backend as workaround
- Not suitable as a drop-in replacement due to missing neutral class

### Winner: lxyuan/distilbert-base-multilingual-cased-sentiments-student
Highest accuracy (58.33%), native 3-class output (positive/negative/neutral),
spread significantly better than rule-based models (0.6446 vs 0.4398).
Originally trained on product review and multilingual B2B data — close to our domain.

## Fine-tuning Cost/Benefit Analysis

### What fine-tuning would mean
Fine-tuning lxyuan (or any DistilBERT variant) on Discovery Lens synthetic data
to better handle PM jargon, B2B SaaS complaints, interview transcripts, and
support tickets.

### Labelling effort
- Minimum viable fine-tuning dataset: 500–1000 labelled examples
- We labelled 60 chunks in ~2 hours including setup time
- Extrapolating: 500 examples ≈ 15–20 hours of manual labelling
- Our synthetic corpus has 1212 chunks total — labelling all would be ~40 hours

### Training cost
- Free on Google Colab T4 GPU
- DistilBERT fine-tune on 500 examples: ~30–60 minutes
- No monetary cost for a one-off run
- Needs to be repeated if synthetic data or source types change significantly

### Maintenance burden
- Model weights need versioning (HuggingFace Hub or repo LFS)
- Owner would be Lucas (owns odi_scorer.py)
- Adds a non-trivial dependency to the pipeline for a capstone project

### What we'd gain
- Potentially 5–15% accuracy improvement over lxyuan on our specific corpus
- Better handling of edge cases: calm professional complaints, mixed-sentiment chunks

### What we already have
- lxyuan at 58.33% — already trained on product review and B2B data
- 20 percentage point improvement over VADER baseline
- Spread of 0.6446 vs VADER's 0.4398 — meaningfully better odi_score discrimination
- Works out of the box, no labelling overhead, no maintenance

### Omar's tip (consulted May 13 2026)
Omar's recommendation: train a Small Language Model (SLM) with encoder architecture
(BERT or T5Gemma) specifically for sentiment classification. Key points:

- Cheaper than a general LLM and likely same performance for this task
- Would need more training data than our current 60 chunks
- Suggested approach: use 60 labelled chunks as seed data, prompt an LLM to generate
  similar synthetic examples, then train the SLM on the expanded dataset
- This is data augmentation via LLM + fine-tuning, significantly reducing the
  manual labelling bottleneck

### Verdict: defer to post-MVP
lxyuan ships in v1. Omar's approach is viable and the groundwork is already done
(60 labelled chunks ready as seed data). Revisit if odi_score discrimination proves
insufficient in end-to-end pipeline testing, or if the project moves to production.

Implementing now would cost 2–3 days on a project past its halfway point, with no
evidence yet that lxyuan is insufficient for the demo.